# 🍷 UTS Data Mining - Wine Quality Classification
**Mata Kuliah:** Data Mining  
**Dosen:** Nur Achmey Selgi Harwanti, S.Stat. M.Stat  
**Prodi:** S1 Pendidikan Matematika - UNNES  
**Semester:** Genap 2025/2026  

---
Notebook ini berisi proses klasifikasi kualitas anggur menggunakan dataset Wine Quality.
Model yang digunakan adalah **Random Forest Classifier** untuk memprediksi nilai `quality` (skala 0-10).

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')
print('Library berhasil diimport!')

## 📂 2. Persiapan & Loading Data

Dataset terdiri dari dua file:
- `data_training.csv` → digunakan untuk melatih model (memiliki kolom `quality`)
- `data_testing.csv` → digunakan untuk prediksi (tanpa kolom `quality`)

Dataset dapat diunduh dari: https://bit.ly/datasetwine

In [ ]:
# Upload file jika menggunakan Google Colab
# from google.colab import files
# uploaded = files.upload()

# Load dataset
train = pd.read_csv('data_training.csv')
test  = pd.read_csv('data_testing.csv')

print(f'Ukuran data training : {train.shape}')
print(f'Ukuran data testing  : {test.shape}')
train.head()

## 🔍 3. Eksplorasi Data (EDA)

Pada tahap ini kita memeriksa:
- Tipe data setiap kolom
- Missing values
- Distribusi variabel target (`quality`)
- Statistik deskriptif

In [ ]:
# Informasi dasar dataset
print('=== INFO DATASET TRAINING ===')
train.info()
print('\n=== STATISTIK DESKRIPTIF ===')
train.describe()

In [ ]:
# Cek missing values
print('Missing values pada data training:')
print(train.isnull().sum())
print(f'\nTotal missing: {train.isnull().sum().sum()}')

print('\nMissing values pada data testing:')
print(test.isnull().sum())
print(f'Total missing: {test.isnull().sum().sum()}')

In [ ]:
# Distribusi variabel target quality
plt.figure(figsize=(8, 4))
quality_counts = train['quality'].value_counts().sort_index()
bars = plt.bar(quality_counts.index, quality_counts.values, color='steelblue', edgecolor='black')
for bar, val in zip(bars, quality_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(val), ha='center', fontweight='bold')
plt.title('Distribusi Kualitas Anggur (Training Data)', fontsize=14)
plt.xlabel('Quality Score')
plt.ylabel('Jumlah Sampel')
plt.xticks(quality_counts.index)
plt.tight_layout()
plt.show()

print('Distribusi quality:')
print(quality_counts)
print('\nCatatan: Dataset cukup imbalanced - mayoritas quality 5 dan 6')

In [ ]:
# Heatmap korelasi fitur
plt.figure(figsize=(12, 8))
corr = train.drop('Id', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Heatmap Korelasi Antar Fitur', fontsize=14)
plt.tight_layout()
plt.show()
print('Fitur dengan korelasi tertinggi terhadap quality:')
print(corr['quality'].drop('quality').abs().sort_values(ascending=False).head(5))

## 🧹 4. Pembersihan & Persiapan Data

Langkah yang dilakukan:
1. Tidak ada missing values → tidak perlu imputasi
2. Memisahkan fitur (X) dan target (y)
3. **Feature Scaling** menggunakan StandardScaler agar semua fitur memiliki skala yang sama
   (penting karena nilai fitur sangat bervariasi, misal `density` sekitar 0.99 vs `total sulfur dioxide` bisa mencapai 200+)

In [ ]:
# Definisi fitur
features = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
            'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
            'density', 'pH', 'sulphates', 'alcohol']

X = train[features]
y = train['quality']
X_test_raw = test[features]
test_ids = test['Id']

print(f'Jumlah fitur: {len(features)}')
print(f'Jumlah sampel training: {len(X)}')
print(f'Jumlah sampel testing: {len(X_test_raw)}')

In [ ]:
# Feature Scaling - StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)        # fit + transform pada training
X_test_scaled = scaler.transform(X_test_raw)  # hanya transform pada testing

print('Sebelum scaling (mean per fitur):')
print(X.mean().round(3))
print('\nSetelah scaling (mean per fitur - mendekati 0):')
print(pd.DataFrame(X_scaled, columns=features).mean().round(3))

## 🤖 5. Pembuatan Model Klasifikasi

Model yang digunakan: **Random Forest Classifier**

Random Forest dipilih karena:
- Robust terhadap data imbalanced (dengan `class_weight='balanced'`)
- Tidak mudah overfitting
- Mampu menangani hubungan non-linear antar fitur
- Memberikan feature importance yang informatif

Data dibagi menjadi 80% training dan 20% validasi untuk mengevaluasi performa model.

In [ ]:
# Split train/validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Data training: {X_tr.shape[0]} sampel')
print(f'Data validasi: {X_val.shape[0]} sampel')

In [ ]:
# Melatih model Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_tr, y_tr)
print('Model berhasil dilatih!')

## 📊 6. Evaluasi Model

Model dievaluasi menggunakan:
- **Accuracy** → persentase prediksi yang benar
- **Classification Report** → precision, recall, f1-score per kelas
- **Confusion Matrix** → visualisasi perbandingan prediksi vs aktual
- **5-Fold Cross Validation** → estimasi performa yang lebih stabil

In [ ]:
# Evaluasi pada data validasi
y_pred_val = rf_model.predict(X_val)
acc = accuracy_score(y_val, y_pred_val)

print(f'Akurasi Validasi: {acc:.4f} ({acc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_val, y_pred_val, zero_division=0))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_val, y_pred_val)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=sorted(y.unique()))
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix - Random Forest (Validasi)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X_scaled, y, cv=cv, scoring='accuracy')

print('Hasil 5-Fold Cross Validation:')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\nRata-rata CV Accuracy : {cv_scores.mean():.4f}')
print(f'Standar Deviasi       : {cv_scores.std():.4f}')
print('\nInterpretasi: Model cukup konsisten antar fold, menunjukkan tidak overfitting.')

In [ ]:
# Feature Importance
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'Fitur': features, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_df['Fitur'], feat_df['Importance'], color='darkorange')
plt.xlabel('Feature Importance')
plt.title('Feature Importance - Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

print('Fitur paling berpengaruh:')
print(feat_df.sort_values('Importance', ascending=False).head(3).to_string(index=False))

## 🔮 7. Prediksi Data Testing

Model di-retrain menggunakan **seluruh data training** (bukan hanya 80%) sebelum digunakan untuk prediksi.
Hal ini dilakukan agar model mendapatkan informasi sebanyak mungkin dari data yang tersedia.

In [ ]:
# Retrain dengan seluruh data training
rf_final = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)
rf_final.fit(X_scaled, y)

# Prediksi data testing
y_test_pred = rf_final.predict(X_test_scaled)

print('Hasil prediksi quality pada data testing:')
pred_series = pd.Series(y_test_pred)
print(pred_series.value_counts().sort_index())

In [ ]:
# Simpan hasil prediksi
hasil = pd.DataFrame({'Id': test_ids, 'Quality': y_test_pred})

# Ganti XXX dengan 3 digit NIM terakhir kamu
nama_file = 'hasilprediksi_XXX.csv'
hasil.to_csv(nama_file, sep=';', index=False)

print(f'File berhasil disimpan: {nama_file}')
print(f'Jumlah baris: {len(hasil)}')
print('\nPreview hasil prediksi:')
hasil.head(10)

In [ ]:
# Download file (Google Colab)
# from google.colab import files
# files.download(nama_file)

## ✅ 8. Kesimpulan

- **Dataset:** 857 sampel training, 286 sampel testing, 11 fitur kimiawi
- **Distribusi target:** Imbalanced (didominasi quality 5 dan 6)
- **Preprocessing:** Tidak ada missing values; dilakukan StandardScaler untuk normalisasi fitur
- **Model:** Random Forest Classifier (200 estimators, class_weight='balanced')
- **Akurasi validasi:** ~64.5%
- **Cross-validation:** ~63.6% ± 3.3%
- **Fitur paling berpengaruh:** alcohol, volatile acidity, sulphates
- **Catatan:** Akurasi 64% sudah wajar untuk dataset wine quality yang sangat imbalanced dan memiliki banyak kelas overlapping (score 5 dan 6 sangat mirip secara kimiawi)